In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Model Training Female: BEFORE SHAP

In [ ]:
import xgboost as xgb
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import RFECV
from imblearn.over_sampling import SMOTE
import warnings

# Ignore all warnings
warnings.simplefilter("ignore")

# Load your dataset
data = pd.read_csv("/kaggle/input/female-data/female_data.csv")
X = data.drop(columns=["Diabetes_Status", "Patient_ID"])
y = data["Diabetes_Status"]

# Split the data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply SMOTE to the training set only
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Initialize evaluation sets
eval_set = [(X_train_resampled, y_train_resampled), (X_val, y_val)]

female_model = xgb.XGBClassifier(eval_metric='merror', objective='multi:softprob', num_class=4)

female_model.fit(X_train_resampled, y_train_resampled, eval_set=eval_set, verbose=True)

# Retrieve evaluation results
results = female_model.evals_result()

# Calculate accuracy
train_accuracy = [1 - x for x in results['validation_0']['merror']]
test_accuracy = [1 - x for x in results['validation_1']['merror']]

# Plot accuracy curve
plt.figure(figsize=(10, 6))
plt.plot(train_accuracy, label='Train Accuracy')
plt.plot(test_accuracy, label='Test Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Accuracy Curve Before SHAP Feature Selection')
plt.legend()
plt.savefig('female_accuracy_curve_before_shap.png')
plt.show()

from sklearn.metrics import classification_report

# Make predictions on the test set
y_prob = female_model.predict_proba(X_val)
# Use np.argmax to convert probabilities to class predictions
y_pred = np.argmax(y_prob, axis=1)

# Generate the classification report
report = classification_report(y_val, y_pred)
print(report)




Calculating SHAP values for female

In [ ]:
import shap
import matplotlib.pyplot as plt
import numpy as np  # Import numpy

# Initialize the SHAP explainer
explainer = shap.TreeExplainer(female_model, X_train_resampled)

# Calculate SHAP values
shap_values = explainer(X_train_resampled)

# Generate and save SHAP summary plot for each class
for i in range(shap_values.values.shape[2]):  # Adjust range if you have more than 4 classes
    plt.figure()  # Start a new figure
    shap.summary_plot(shap_values[..., i], X_train_resampled, show=False)
    plt.title(f'SHAP Summary for Class {i}')  # Optional: Add title
    plt.savefig(f'female_shap_summary_class_{i}.png')  # Save plot
    plt.close()  # Close the figure to avoid display overlap

# Calculate mean absolute SHAP values across classes for each feature
mean_shap_values = np.mean(np.abs(shap_values.values), axis=2)  # Corrected aggregation method

# Plot and save overall SHAP summary plot
plt.figure()
shap.summary_plot(mean_shap_values, X_train_resampled, show=False)
plt.title('SHAP Summary for All Classes')
plt.savefig('female_shap_summary_all_classes.png')
plt.close()

Choosing new features for female model based on SHAP and retrain

In [ ]:
female_feature_importance = np.mean(np.abs(shap_values.values), axis=(0, 2))  # For multiclass
# Rank features and select the top ones
num_features_to_keep = 10  # Choose based on your needs
top_features_indices = np.argsort(female_feature_importance)[-num_features_to_keep:]
top_features = X_train_resampled.columns[top_features_indices]

print(f"Top Features Female: {top_features}")

X_train_reduced = X_train_resampled[top_features]
X_val_reduced = X_val[top_features]

# Initialize evaluation sets
eval_set = [(X_train_reduced, y_train_resampled), (X_val_reduced, y_val)]

female_model_retrain = xgb.XGBClassifier(eval_metric='merror', objective='multi:softprob', num_class=4)

female_model_retrain.fit(X_train_reduced, y_train_resampled, eval_set=eval_set, verbose=True)

# Retrieve evaluation results
results = female_model_retrain.evals_result()

# Calculate accuracy
train_accuracy = [1 - x for x in results['validation_0']['merror']]
test_accuracy = [1 - x for x in results['validation_1']['merror']]

# Plot accuracy curve
plt.figure(figsize=(10, 6))
plt.plot(train_accuracy, label='Train Accuracy')
plt.plot(test_accuracy, label='Test Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Accuracy Curve After SHAP Feature Selection')
plt.legend()
plt.savefig('female_accuracy_curve_after_shap.png')
plt.show()


Classification Report: Female

In [ ]:
from sklearn.metrics import classification_report

# Make predictions on the test set
y_prob = female_model_retrain.predict_proba(X_val_reduced)
# Use np.argmax to convert probabilities to class predictions
y_pred = np.argmax(y_prob, axis=1)

# Generate the classification report
report = classification_report(y_val, y_pred)
print(report)

Model Training Male

In [ ]:
import xgboost as xgb
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import RFECV
from imblearn.over_sampling import SMOTE
import warnings

# Ignore all warnings
warnings.simplefilter("ignore")

# Load your dataset
data = pd.read_csv("/kaggle/input/male-data/male_data.csv")
X = data.drop(columns=["Diabetes_Status", "Patient_ID"])
y = data["Diabetes_Status"]

# Split the data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply SMOTE to the training set only
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Initialize evaluation sets
eval_set = [(X_train_resampled, y_train_resampled), (X_val, y_val)]

male_model = xgb.XGBClassifier(eval_metric='merror', objective='multi:softprob', num_class=3)

male_model.fit(X_train_resampled, y_train_resampled, eval_set=eval_set, verbose=True)

# Retrieve evaluation results
results = male_model.evals_result()

# Calculate accuracy
train_accuracy = [1 - x for x in results['validation_0']['merror']]
test_accuracy = [1 - x for x in results['validation_1']['merror']]

# Plot accuracy curve
plt.figure(figsize=(10, 6))
plt.plot(train_accuracy, label='Train Accuracy')
plt.plot(test_accuracy, label='Test Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Accuracy Curve Before SHAP Feature Selection')
plt.legend()
plt.savefig('male_accuracy_curve_before_shap.png')
plt.show()

from sklearn.metrics import classification_report

# Make predictions on the test set
y_prob = male_model.predict_proba(X_val)
# Use np.argmax to convert probabilities to class predictions
y_pred = np.argmax(y_prob, axis=1)

# Generate the classification report
report = classification_report(y_val, y_pred)
print(report)




Calculating SHAP values for male

In [ ]:
import shap
import matplotlib.pyplot as plt
import numpy as np  # Import numpy

# Initialize the SHAP explainer
explainer = shap.TreeExplainer(male_model, X_train_resampled)

# Calculate SHAP values
shap_values = explainer(X_train_resampled)

# Generate and save SHAP summary plot for each class
for i in range(shap_values.values.shape[2]):  # Adjust range if you have more than 4 classes
    plt.figure()  # Start a new figure
    shap.summary_plot(shap_values[..., i], X_train_resampled, show=False)
    plt.title(f'SHAP Summary for Class {i}')  # Optional: Add title
    plt.savefig(f'male_shap_summary_class_{i}.png')  # Save plot
    plt.close()  # Close the figure to avoid display overlap

# Calculate mean absolute SHAP values across classes for each feature
mean_shap_values = np.mean(np.abs(shap_values.values), axis=2)  # Corrected aggregation method

# Plot and save overall SHAP summary plot
plt.figure()
shap.summary_plot(mean_shap_values, X_train_resampled, show=False)
plt.title('SHAP Summary for All Classes')
plt.savefig('male_shap_summary_all_classes.png')
plt.close()

In [ ]:
male_feature_importance = np.mean(np.abs(shap_values.values), axis=(0, 2))  # For multiclass
# Rank features and select the top ones
num_features_to_keep = 10  # Choose based on your needs
top_features_indices = np.argsort(male_feature_importance)[-num_features_to_keep:]
top_features = X_train_resampled.columns[top_features_indices]

print(f"Top Features Male: {top_features}")

X_train_reduced = X_train_resampled[top_features]
X_val_reduced = X_val[top_features]

# Initialize evaluation sets
eval_set = [(X_train_reduced, y_train_resampled), (X_val_reduced, y_val)]

male_model_retrain = xgb.XGBClassifier(eval_metric='merror', objective='multi:softprob', num_class=3)

male_model_retrain.fit(X_train_reduced, y_train_resampled, eval_set=eval_set, verbose=True)

# Retrieve evaluation results
results = male_model_retrain.evals_result()

# Calculate accuracy
train_accuracy = [1 - x for x in results['validation_0']['merror']]
test_accuracy = [1 - x for x in results['validation_1']['merror']]

# Plot accuracy curve
plt.figure(figsize=(10, 6))
plt.plot(train_accuracy, label='Train Accuracy')
plt.plot(test_accuracy, label='Test Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Accuracy Curve After SHAP Feature Selection')
plt.legend()
plt.savefig('male_accuracy_curve_after_shap.png')
plt.show()


Classification Report: Male

In [ ]:
from sklearn.metrics import classification_report

# Make predictions on the test set
y_prob = male_model_retrain.predict_proba(X_val_reduced)
# Use np.argmax to convert probabilities to class predictions
y_pred = np.argmax(y_prob, axis=1)

# Generate the classification report
report = classification_report(y_val, y_pred)
print(report)

In [ ]:
import xgboost as xgb

male_model_retrain.save_model('xgboost_male.bin')
female_model_retrain.save_model('xgboost_female.bin')